# 04 — Language Models (Module B)
**NewsBot Intelligence System 2.0 | ITAI 2373**

LLM integration via ollama/llama3.2:
- B.1 Intelligent Summarization
- B.2 Content Enhancement
- B.3 Article Q&A
- B.4 Insight Generation

In [ ]:
!pip install ollama requests -q

In [ ]:
import requests, json, time
from IPython.display import Markdown, display

OLLAMA_HOST  = 'http://127.0.0.1:11434'
OLLAMA_MODEL = 'llama3.2:1b'   # change if you have a different model
TEMPERATURE  = 0.3
MAX_TOKENS   = 512

def check_ollama():
    try:
        r = requests.get(OLLAMA_HOST, timeout=5)
        print(f'ollama status: {r.text.strip()}')
        return True
    except Exception as e:
        print(f'ollama not reachable: {e}')
        print('Run: ollama serve')
        return False

check_ollama()


In [ ]:
def call_llm(prompt, system=None):
    """Call ollama via HTTP API."""
    messages = []
    if system:
        messages.append({'role':'system','content':system})
    messages.append({'role':'user','content':prompt})

    r = requests.post(
        f'{OLLAMA_HOST}/api/chat',
        json={'model':OLLAMA_MODEL,'messages':messages,
              'stream':False,'options':{'temperature':TEMPERATURE,'num_predict':MAX_TOKENS}},
        timeout=120
    )
    r.raise_for_status()
    return r.json()['message']['content'].strip()

def extract_json(text):
    """Parse JSON from LLM response, stripping markdown fences."""
    clean = text.strip().lstrip('```json').lstrip('```').rstrip('```').strip()
    try:
        return json.loads(clean)
    except:
        return {'raw_response': text}

print('LLM helpers ready.')


In [ ]:
# ── Verify df_final ───────────────────────────────────────────────
try:
    _ = df_final
    print(f'df_final: {len(df_final)} articles')
except NameError:
    raise RuntimeError('Run notebooks 01 and 02 first.')


## B.1 — Intelligent Summarization

In [ ]:
def generate_summary(article_text, max_sentences=3, category=None):
    system = ('You are a professional news editor. Write accurate concise summaries. '
              'Preserve all named entities exactly as written.')
    cat_hint = f' This is a {category} article.' if category else ''
    prompt = (f'Summarize this news article in exactly {max_sentences} sentences.{cat_hint} '
              f'Cover Who, What, When, Where, Why. Output only the summary, no preamble.\n\n'
              f'ARTICLE:\n{article_text[:3000]}')
    summary = call_llm(prompt, system)
    orig_words    = len(article_text.split())
    summary_words = len(summary.split())
    return {
        'summary':           summary,
        'word_count':        summary_words,
        'compression_ratio': round(1 - summary_words/max(orig_words,1), 3)
    }

# Demo on one article per category
print('B.1 — Summarization Demo\n' + '='*55)
for cat in ['tech','business','politics']:
    sample = df_final[df_final.category==cat].sample(1, random_state=42).iloc[0]
    result = generate_summary(sample.text, max_sentences=3, category=cat)
    print(f'\n[{cat.upper()}]')
    print(f'Summary: {result["summary"]}')
    print(f'Words: {result["word_count"]} | Compression: {result["compression_ratio"]*100:.0f}%')


## B.2 — Content Enhancement

In [ ]:
def enhance_content(article_text, category=None, entities=None):
    entity_hint = ''
    if entities:
        all_ents = [e for lst in entities.values() for e in lst[:3]]
        if all_ents:
            entity_hint = f' Key entities: {chr(44).join(all_ents[:5])}.'
    system = ('You are a senior news analyst. '
              'Respond ONLY with valid JSON. No markdown, no preamble.')
    prompt = (f'Analyze this news article and return JSON with keys:\n'
              f'  background_context (2-3 sentences of relevant background)\n'
              f'  related_trends     (2-3 sentences on broader trends)\n'
              f'  implications       (2-3 sentences on potential impact)\n'
              f'  entities_to_watch  (list of 3-5 entity name strings)\n\n'
              f'Article:{" Category: "+category if category else ""}{entity_hint}\n{article_text[:2500]}')
    raw = call_llm(prompt, system)
    return extract_json(raw)

# Demo
sample = df_final[df_final.category=='tech'].sample(1, random_state=1).iloc[0]
result = enhance_content(sample.text, category='tech', entities=sample.get('entities',{}))
print('B.2 — Content Enhancement Demo')
print('='*55)
for key in ['background_context','related_trends','implications']:
    if key in result:
        print(f'\n{key.upper().replace("_"," ")}:')
        print(result[key])
if 'entities_to_watch' in result:
    print(f'\nENTITIES TO WATCH: {result["entities_to_watch"]}')


## B.3 — Article Q&A (Multi-turn)

In [ ]:
class ArticleQueryEngine:
    """Stateful multi-turn Q&A engine for a single article."""
    SYSTEM = ('You are NewsBot 2.0, an expert AI news analyst. '
              'Answer questions about news articles clearly and concisely. '
              'Ground responses in the provided article context.')

    def __init__(self, article_text, category=None):
        self.article_text = article_text
        self.category     = category
        self._history     = []
        cat_str = f'[{category.upper()}] ' if category else ''
        self._context = f'{self.SYSTEM}\n\nArticle: {cat_str}\n{article_text[:3000]}'

    def ask(self, question):
        history_str = ''
        if self._history:
            history_str = '\n'.join(
                f'Q: {t["question"]}\nA: {t["answer"]}' for t in self._history[-4:]
            )
            history_str = f'\nConversation history:\n{history_str}\n'
        prompt = f'{self._context}{history_str}\nQuestion: {question}\nAnswer:'
        answer = call_llm(prompt)
        self._history.append({'question':question,'answer':answer})
        return answer

    def show_history(self):
        for i,t in enumerate(self._history,1):
            print(f'[Q{i}] {t["question"]}')
            print(f'[A{i}] {t["answer"]}\n')

    def reset(self): self._history = []

# Demo — multi-turn conversation
print('B.3 — Multi-turn Q&A Demo')
print('='*55)
sample = df_final[df_final.category=='politics'].sample(1, random_state=5).iloc[0]
print(f'Article preview: {sample.text[:200]}...\n')

engine = ArticleQueryEngine(sample.text, category='politics')
questions = [
    'What is the main policy being discussed?',
    'Who are the key people or parties involved?',
    'What are the potential implications of this?'
]
for q in questions:
    print(f'Q: {q}')
    a = engine.ask(q)
    print(f'A: {a}\n')


## B.4 — Insight Generation

In [ ]:
def generate_insights(article_text, nlp_metadata=None):
    meta_str = ''
    if nlp_metadata:
        parts = []
        if 'sentiment_label' in nlp_metadata:
            parts.append(f'Sentiment: {nlp_metadata["sentiment_label"]} ({nlp_metadata.get("sentiment_compound","N/A")})')
        if 'top_tfidf' in nlp_metadata:
            terms = [t for t,_ in nlp_metadata['top_tfidf'][:5]]
            parts.append(f'Top TF-IDF: {chr(44).join(terms)}')
        meta_str = '\n'.join(parts)

    system = ('You are an expert NLP analyst. '
              'Respond ONLY with valid JSON. Ground all observations in the article.')
    prompt = (f'Analyze this article and return JSON with keys:\n'
              f'  key_findings         (list of 3-5 finding strings)\n'
              f'  patterns             (list of 2-3 pattern strings)\n'
              f'  entities_of_interest (list of entity name strings)\n'
              f'  sentiment_drivers    (list of 2-3 phrases driving sentiment)\n'
              f'  recommended_queries  (list of 4 natural language questions)\n\n'
              f'NLP metadata:\n{meta_str}\n\nArticle:\n{article_text[:2500]}')
    raw = call_llm(prompt, system)
    return extract_json(raw)

# Demo
sample = df_final[df_final.category=='business'].sample(1, random_state=10).iloc[0]
meta = {
    'sentiment_label':    sample.get('sentiment_label',''),
    'sentiment_compound': sample.get('sentiment_compound',''),
    'top_tfidf':          sample.get('top_tfidf',[])
}
result = generate_insights(sample.text, nlp_metadata=meta)

print('B.4 — Insight Generation Demo')
print('='*55)
for key in ['key_findings','patterns','sentiment_drivers','recommended_queries']:
    if key in result:
        print(f'\n{key.upper().replace("_"," ")}:')
        items = result[key] if isinstance(result[key], list) else [result[key]]
        for item in items:
            print(f'  - {item}')


## B.5 — Batch Pipeline

In [ ]:
def run_module_b(df, n_samples=5, random_state=42):
    """Run full Module B pipeline on a sample of articles."""
    sample = df.groupby('category').apply(
        lambda g: g.sample(min(1, len(g)), random_state=random_state)
    ).reset_index(drop=True)
    if n_samples:
        sample = sample.head(n_samples)

    results = []
    for _, row in sample.iterrows():
        print(f'Processing [{row.category}]...')
        entry = {'doc_id': row.get('doc_id', _), 'category': row.category}
        try:
            entry['summary'] = generate_summary(row.text, category=row.category)
        except Exception as e:
            entry['summary'] = {'error': str(e)}
        try:
            meta = {'sentiment_label': row.get('sentiment_label',''),
                    'sentiment_compound': row.get('sentiment_compound',0),
                    'top_tfidf': row.get('top_tfidf',[])}
            entry['insights'] = generate_insights(row.text, nlp_metadata=meta)
        except Exception as e:
            entry['insights'] = {'error': str(e)}
        results.append(entry)
        time.sleep(0.5)

    return results

print('Running Module B batch pipeline (1 article per category)...')
batch_results = run_module_b(df_final, n_samples=5)
print(f'\nProcessed {len(batch_results)} articles.')
for r in batch_results:
    s = r.get('summary',{})
    print(f'  [{r["category"]}] Summary: {s.get("word_count","?")} words, compression {s.get("compression_ratio","?")}')
